# 🫀 퀘스트 46 · Q4-C — **학습 때의 burden 인가, 테스트 때의 burden 인가**

| | **MedKOS / `notebooks/quest46_q4c_burden_train_vs_test.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층② 점수 눈금 |
| 부모 런 | `quest46_q4b_burden_feature_v2`(`20260805T0114`) · `quest46_q4_burden_feature` |
| 성격 | **분해 + 지표 교체** — 이득의 출처를 가르고, 검정력 없는 지표를 내린다 |

## Q4-B 가 남긴 것

**주 관문은 통과했다** — `B_add − A` 매크로 **+0.0582** [+0.0318, +0.0867] (측정된
영점 −0.0290 [−0.0544, −0.0059] · 문턱을 0 으로 둬도 통과). 방법 A 는 레코드별 상수
시프트라 매크로가 **정의상 불변**이고(E0 max|Δ| = 0.0e+00) 이건 **A 가 원리적으로
못 하는 일**이다.

★★★ 그런데 **π̂ 오차가 두 지표에 비대칭으로 든다**:

```
B_add_em − B_add_oracle      매크로 −0.0035        전역 −0.1588        (45배)
```

`add` 모드에서 burden 은 **레코드 내 상수**이고 선형으로 들어간다. 그러면 테스트시
burden 은 그 레코드의 로짓을 **통째로 평행이동**시킬 뿐이라 **레코드 내 순위를 못
바꾼다**. 즉 매크로 이득은 **「학습 때 올바른 burden 을 썼다」의 몫**이고 π̂ 를
**필요로 하지 않는다** — Q3 이 막혔던 병목을 레코드 내 지표에서는 **우회한다**.

⚠️ 하지만 Q4-B 는 이걸 **증명하지 못했다**. `B_add_shuf` 가 학습과 테스트를 **동시에**
뒤섞어서, 매크로 차 +0.0538 이 학습 몫인지 테스트 몫인지 갈리지 않는다. **빠진 칸이
`학습 true / 테스트 shuf` 다.**

## 무엇을 바꾸나

### ① 2×2 로 분해한다 — 빠진 칸을 채운다

| | 테스트 `true` | 테스트 `em` | 테스트 `shuf` |
|---|---|---|---|
| **학습 `true`** | `TT` (= Q4-B `B_add_oracle`) | `TE` (= `B_add_em`) | **`TS` ← 새 칸** |
| **학습 `shuf`** | **`ST` ← 새 칸** | — | `SS` (= `B_add_shuf`) |

- **학습 몫** = `TS − SS` (테스트를 똑같이 틀리게 두고 학습만 고친다)
- **테스트 몫** = `TT − TS` (학습을 똑같이 옳게 두고 테스트만 고친다)
- 가법성 검산 — 두 몫의 합이 `TT − SS` 와 맞는가

### ② 전역 PR-AUC 를 관문에서 내리고 **교차레코드 AUROC** 로 갈아탄다

Q4-B 의 E3 은 미결이었는데 **표본을 늘려서 될 일이 아니다** — 문턱 0 기준 필요표본이
**80% 검정력에서 356 레코드**인데 SVDB 채점 가능 상한은 **56** 이다. 유병률이
0.0070~0.5764 로 82배 벌어져 있어 레코드 하나(48)가 양성의 **15.2%** 를 갖는 탓에,
레코드를 더해도 √n 이 안 먹는다(Q4-B 실측 MDE 0.1614 → 0.1365, 1.18배뿐).

전역이 매크로에 **더해 주는 정보는 「레코드 간 척도 정렬」 하나**다. 그러면 그것만
직접 재면 된다 — **교차레코드 AUROC**: 레코드 쌍 (i,j) 마다 「i 의 양성 > j 의 음성」
비율을 재고 **쌍마다 동일 가중**으로 평균한다. 지배 지분에 안 눌린다(레코드 48 의
가중이 15.2% → 3.6%). 전역은 **상한과 함께 보고만** 한다(R36 ①).

### ③ 배포 가능판끼리를 관문으로 승격한다

**오라클은 상한이지 방법이 아니다.** Q4-B 실측 `A_em` 전역 **0.1254** 는 raw(0.2097)
**보다도 나쁘다** — 방법 A 는 배포 불가다. `TE − A_em` 이 결정 관련 대비인데
Q4-B 는 이걸 「참고」로 뒀다(양 지표 모두 CI 가 0 을 뗐는데도).

### ④ Q4-B 의 자체 오류 둘을 고친다

- **필요표본을 관문 문턱 기준으로** — Q4-B 는 **영점 평균** 기준으로 계산해 E3 을
  「n(80%)=32 · 읽을 수 있다」로 적었다. 관문 기준으론 **356** 이다(R40 ②)
- **문턱 규칙 통일** — Q4-B 는 E2 에 `영점 상단`, E3 에 `max(0, 영점 상단)` 을 썼다.
  둘 다 우월 판정이므로 **`max(0, 영점 상단)`** 으로 통일한다

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **F0** | 코호트 · 항등 — A 팔 매크로 ≡ raw 매크로 | 구성 항등. 깨지면 **중단** |
| **F1 ★★★ 자** | **보정 전** `TT` 와 `TS` 의 레코드 내 매크로가 **정확히 같은가** | 구성 항등(레코드 내 상수 시프트). 깨지면 **중단** |
| **F2 ★★★ 공동 주** | `TT − A_oracle` **매크로** | max(0, 측정된 영점 상단) 초과 |
| **F3 ★★★ 공동 주** | `TT − A_oracle` **교차레코드 AUROC** | max(0, 측정된 영점 상단) 초과 |
| **F4 ★★★ 분해** | 학습 몫 `TS − SS` vs 테스트 몫 `TT − TS` (양 지표) | 관문 아님. **분해를 보고** |
| **F5 ★★ 배포** | `TE − A_em` (양 지표) | max(0, 측정된 영점 상단) 초과 |
| **F6** | 전역 PR-AUC — **관문 아님** | 상한 + 관문 기준 필요표본만 보고 |
| **F7** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **F2 ✅ · F3 ✅** → B 가 **레코드 내·레코드 간 모두** A 를 이긴다
- **F2 ✅ · F3 미결/❌** → 이득은 **레코드 내 판별**에 한정된다. 척도 정렬은 미확립
- **F4 에서 테스트 몫 ≈ 0** → **π̂ 없이 쓸 수 있는 처방**이다(Q3 병목 우회)
- **F4 에서 테스트 몫이 크다** → π̂ 품질이 여전히 병목 — Q3 갈래를 다시 연다

⚠️ **새 데이터 0** — `svdb_data5.npz` 만.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    """★ **짝지은 차**(b − a) — 같은 레코드에서 두 팔을 재므로 짝을 유지한다."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def _rank_avg(v):
    v = np.asarray(v, float); o = v.argsort()
    r = np.empty(len(v), float); r[o] = np.arange(len(v), dtype=float)
    for u in np.unique(v):
        m = v == u
        if m.sum() > 1:
            r[m] = r[m].mean()
    return r

def spearman(a, b):
    """★★ 동점을 **평균 순위**로(Q3-B 에서 argsort 판본의 순서 의존이 드러났다)."""
    ra, rb = _rank_avg(a), _rank_avg(b)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def need_super(n, half, eff, p80=False):
    """★★ Q4-B 오류 정정 — `eff` 는 **관문 문턱과의 거리**여야 한다(영점 평균이 아니라).
    Q4-B 는 영점 평균 기준으로 재서 E3 을 「n(80%)=32」로 적었는데, 관문 기준으론
    **356** 이었다. 관문이 묻는 질문과 필요표본이 답하는 질문이 달랐다(R40 ②)."""
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

def derangement(n, rng):
    for _ in range(1000):
        p = rng.permutation(n)
        if not np.any(p == np.arange(n)):
            return p
    return np.roll(np.arange(n), 1)

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
FS = 360
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25

# ── ★ 사전등록 상수 (SMOKE 가 절대 안 건드린다)
TOL_IDENT = 1e-12
DEV_EVERY = 4

# ── 비용 손잡이
NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 2   if SMOKE else 5

# ★★★ 2×2 분해 — 빠진 칸(`TS`·`ST`)을 채운다.
#     이름은 (학습 burden, 테스트 burden) 이다.
BSPEC = {"TT": ("true", "true"),    # = Q4-B `B_add_oracle`
         "TE": ("true", "em"),      # = Q4-B `B_add_em`  (배포 가능판)
         "TS": ("true", "shuf"),    # ★ 새 칸 — 테스트 몫만 떼어 낸다
         "ST": ("shuf", "true"),    # ★ 새 칸 — 학습 몫만 떼어 낸다
         "SS": ("shuf", "shuf")}    # = Q4-B `B_add_shuf`
ARMS = ("raw", "A_oracle", "A_em") + tuple(BSPEC)

# ★★ 주 지표에서 **전역을 뺐다** — SVDB 에서 구조적으로 검정력이 없다.
#    대신 교차레코드 AUROC(쌍마다 동일 가중)를 공동 주 지표로 올린다.
PRIMARY = ("macro", "xrec")
REPORT_ONLY = ("pooled",)
READ_ORDER = ("F0", "F1", "F2", "F3", "F4", "F5", "F6", "F7")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(  # Q4-B(`20260805T0114`) 실측 — 같은 LORO 코호트라 **재현 앵커로 쓴다**
    n_ok=56, dom_rec=48, dom_share=0.152,
    q4b_pool_raw=0.2097, q4b_pool_Aor=0.4495, q4b_pool_Aem=0.1254,
    q4b_pool_TT=0.5268, q4b_pool_TE=0.3680, q4b_pool_SS=0.2035,
    q4b_macro_raw=0.5117, q4b_macro_TT=0.5698, q4b_macro_TE=0.5663,
    q4b_macro_SS=0.5160,
    q4b_E2=0.0582, q4b_E2_lo=0.0318, q4b_E2_hi=0.0867,
    q4b_E3=0.0773, q4b_E3_lo=-0.0723, q4b_E3_hi=0.2008, q4b_E3_mde=0.1365,
    q4b_rho_TT=0.784555, q4b_contrib=3.335e-16,
    q4b_dom48_raw=0.6169, q4b_dom48_TT=0.8034, q4b_dom48_TE=0.8474,
    q4b_need_E3_gate=356)   # ★ Q4-B 가 32 로 잘못 적은 그 수(관문 기준 재계산)

RULE_CHECK = {
    "R11 매크로":       "★★★ 매크로가 공동 주 지표. **전역은 관문에서 내렸다**(검정력 없음)",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "LORO 안에서 기저·보정을 **held-out 레코드를 빼고** 적합",
    "R26 / R38 ②":      "★★ 대비의 **영점**을 rep×레코드로 측정한다. 못 쟀으면 **안 읽는다**",
    "R29 ② 분기 금지":   "F0 · F1 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ③ 대조":       "★★ `shuf` 는 같은 burden 값 집합, **대응만** 깨진다(derangement)",
    "R35 ① 자 먼저":    "★★★ **F1 이 이 런의 자다** — 보정 전 `TT` ≡ `TS` 를 먼저 세운다",
    "R36 ① 접기/닫기":  "★ 전역은 **접는다**(상한 + 조건부 재개), 닫지 않는다",
    "R36 ⑤ 성분":       "차와 함께 **성분**을 보고한다",
    "R39 ① 여유 고정":  "문턱 규칙을 **max(0, 영점 상단)** 으로 사전 고정(양 관문 동일)",
    "R40 ② 같은 통계":  "★★★ **필요표본을 관문 문턱 기준으로** — Q4-B 는 영점 평균 기준으로 "
                        "재서 E3 을 「32」로 적었다(관문 기준 **356**)",
}

CONFIG = dict(
    exp="quest46_q4c_burden_train_vs_test", quest="ailab-2026-0046",
    step="burden-train-vs-test",
    parent_exp=["quest46_q4b_burden_feature_v2", "quest46_q4_burden_feature"],
    purpose=("**이득의 출처를 가르고, 검정력 없는 지표를 내린다.** Q4-B(`20260805T0114`)가 "
             "주 관문을 통과시켰다(`B_add − A` 매크로 **+0.0582** [+0.0318, +0.0867] · "
             "A 는 상수 시프트라 매크로가 정의상 불변이므로 **A 가 원리적으로 못 하는 일**). "
             "★★★ 그런데 π̂ 오차가 **비대칭**으로 든다 — `B_add_em − B_add_oracle` 이 "
             "매크로 **−0.0035** 인데 전역 **−0.1588** 이다(45배). `add` 모드에서 burden 은 "
             "레코드 내 상수라 테스트시 burden 이 로짓을 **통째로 평행이동**시킬 뿐 레코드 내 "
             "순위를 못 바꾸기 때문이다. 즉 매크로 이득은 **「학습 때 올바른 burden 을 썼다」의 "
             "몫**이고 π̂ 를 **필요로 하지 않을** 수 있다 — Q3 병목의 우회다. "
             "⚠️ 그런데 Q4-B 는 이걸 **증명하지 못했다**: `B_add_shuf` 가 학습과 테스트를 "
             "**동시에** 뒤섞어서 두 몫이 안 갈린다. **빠진 칸이 `학습 true / 테스트 shuf`** 다. "
             "이 런은 **2×2**(학습 {true,shuf} × 테스트 {true,em,shuf})로 분해하고, "
             "**전역 PR-AUC 를 관문에서 내려**(문턱 0 기준 필요표본 **356 레코드** · 가용 56) "
             "**교차레코드 AUROC**(쌍마다 동일 가중)로 갈아타며, **배포 가능판끼리"
             "(`TE − A_em`)를 관문으로 승격**한다(Q4-B 실측 `A_em` 전역 0.1254 는 "
             "raw 0.2097 보다도 나쁘다 — 방법 A 는 배포 불가)."),
    dataset="SVDB — svdb_data5.npz (리듬 특징만 · 새 데이터 0)",
    arms=list(ARMS), bspec={k: list(v) for k, v in BSPEC.items()},
    primary=list(PRIMARY), report_only=list(REPORT_ONLY), read_order=READ_ORDER,
    dev_every=DEV_EVERY, n_boot=NB_BOOT, n_perm=N_PERM, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "F0": "코호트 + **항등** — A 팔은 레코드별 상수 시프트라 매크로가 raw 와 "
              "**정확히 같아야** 한다. 깨지면 **중단**",
        "F1": "★★★ **이 런의 자(R35 ①)** — **보정 전** `TT` 와 `TS` 의 레코드 내 매크로가 "
              "**정확히 같아야** 한다. burden 은 레코드 내 상수이고 선형으로 들어가므로 "
              "테스트시 burden 은 **순전한 상수 시프트**다. 깨지면 구현이 틀린 것이므로 "
              "F4 를 **읽지 않는다**",
        "F2": "★★★ **공동 주 관문(매크로)** — `TT − A_oracle`. Q4-B 재현 앵커 +0.0582",
        "F3": "★★★ **공동 주 관문(교차레코드 AUROC)** — `TT − A_oracle`. 매크로가 못 보는 "
              "**레코드 간 척도 정렬**만 잰다. 쌍마다 동일 가중이라 지배 지분(0.152)에 "
              "안 눌린다",
        "F4": "★★★ **분해(관문 아님)** — 학습 몫 `TS − SS` vs 테스트 몫 `TT − TS`. "
              "사전등록 예측: **매크로의 테스트 몫 ≈ 0**(보정 전엔 F1 로 정확히 0 이고, "
              "보정 후엔 **등장성 동점 구조**로만 샌다), **교차레코드의 테스트 몫은 크다**. "
              "가법성(두 몫의 합 = `TT − SS`)도 함께 검산한다",
        "F5": "★★ **배포 가능판끼리** `TE − A_em` — 오라클은 **상한이지 방법이 아니다**. "
              "Q4-B 실측 `A_em` 전역 0.1254 < raw 0.2097 이라 방법 A 는 배포 불가",
        "F6": "전역 PR-AUC — **관문 아님**. 상한과 **관문 기준** 필요표본만 보고(R36 ①)",
        "F7": "결론 검산표"},
    caveat=("★★★ **F1 이 F4 의 전제다** — 보정 전 항등이 안 서면 「테스트 몫 ≈ 0」을 "
            "등장성 탓으로 돌릴 수 없다. 그래서 F1 을 **런타임에 검사하고 깨지면 중단**한다. "
            "★★ **등장성 몫이 작다고 보장된 게 아니다** — π̂ 가 보정 지지구간 밖으로 밀리면 "
            "예측이 상수로 붕괴할 수 있다(합성 재현에서 고유값 14 → 1). Q4-B 의 −0.0035 는 "
            "**이번 π̂ 에서 작았을 뿐**이므로 `TS` 로 **상한을 실측**한다. "
            "★ **전역을 관문에서 내린 이유는 결과가 나빠서가 아니다** — Q4-B 의 E3 은 "
            "미결이었고 표본을 늘려서 될 일이 아니다(관문 기준 356 레코드 · 가용 56). "
            "닫는 게 아니라 **접는다**: 상한 +0.2008 을 남기고 코호트가 커지면 재개한다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4c_burden_train_vs_test", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-C — 학습 때의 burden 인가, 테스트 때의 burden 인가**")
run.log("  ★★★ **2×2 분해** — 빠진 칸 `TS`(학습 true/테스트 shuf)·`ST`(학습 shuf/테스트 true)")
run.log("  ★★★ **전역을 관문에서 내리고 교차레코드 AUROC 로** — 전역은 관문 기준 "
        f"**{REF['q4b_need_E3_gate']} 레코드**가 필요한데 가용 {REF['n_ok']} 이다")
run.log("  ★★ **배포 가능판끼리(`TE − A_em`)를 관문으로 승격** — 오라클은 상한이지 방법이 아니다")
run.log("  ★ Q4-B 오류 둘 정정 — 필요표본을 **관문 문턱 기준**으로 · 문턱 규칙 **통일**")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM}). "
            "관문 문턱은 그대로다")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【F-0】 코호트 · LORO 골격 · 2×2 팔 정의 · 교차레코드 AUROC
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score

run.log("\n" + "=" * 100)
run.log("【F-0】 코호트 · LORO 골격 · 2×2 팔 정의")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT_ = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))
RHY = np.nan_to_num(np.c_[f1, np.column_stack([f2[k] for k in RHY_K]), f3, f4,
                          np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                    nan=0.0, posinf=0.0, neginf=0.0)

IDXS = {int(r): np.where(RID == r)[0] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
s_all = np.array([int(TT_[IDXS[r]].sum()) for r in REC_OK], float)
DOMINANT = float(s_all.max() / s_all.sum())
DOM_REC = int(REC_OK[int(np.argmax(s_all))])
NRE = len(REC_OK)
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}** · 제외 {len(RS)-NRE}")
run.log(f"  유병률 {min(BURD.values()):.4f}~{max(BURD.values()):.4f} · "
        f"지배 지분 **{DOMINANT:.3f}**(레코드 {DOM_REC})")
run.log(f"  ★★ 교차레코드 AUROC 는 **쌍마다 동일 가중**이라 지배 지분이 "
        f"{DOMINANT:.3f} → **{1.0/NRE:.3f}** 로 내려간다")

def make_iso(s, y):
    ir = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6)
    ir.fit(np.asarray(s), np.asarray(y).astype(float))
    return lambda v: np.clip(ir.predict(np.asarray(v)), 1e-6, 1 - 1e-6)

EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

def em_prior(p, pi_tr, iters=100, tol=1e-9, clip=1e-2):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    tr = [r for r in rest if r not in set(dv)]
    return tr, dv

# ── ★★★ 2×2 — 학습 burden 과 테스트 burden 을 **따로** 받는다.
#    Q4-B 의 `B_add_shuf` 는 둘을 동시에 뒤섞어 두 몫이 안 갈렸다.
def loro_B(train_src, test_src, shuf_map, y_override=None, want_precal=False):
    out = np.full(len(K), np.nan)
    pre_ = np.full(len(K), np.nan) if want_precal else None
    for held in REC_OK:
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        bmap = BURD if train_src == "true" else shuf_map
        mu = float(np.mean([bmap[r] for r in tr_r]))
        bv = lambda ii: np.array([bmap[int(r)] for r in RID[ii]], float)
        Ftr = np.c_[RHY[tr], bv(tr)]
        fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        ytr = TT_[tr].astype(int) if y_override is None else y_override[held]
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
        sc = lambda ii, b_: lr.decision_function((np.c_[RHY[ii], b_] - fmu) / fsd)
        cal = make_iso(sc(dv, bv(dv)), TT_[dv])
        pi_tr = float(TT_[dv].mean())
        if test_src == "true":
            bh = BURD[held]
        elif test_src == "shuf":
            bh = shuf_map[held]
        else:                        # em — 무라벨 추정으로 held-out burden 을 채운다
            p0 = np.clip(cal(sc(te, np.full(len(te), mu))), EPS, 1 - EPS)
            bh = em_prior(p0, pi_tr)
        s_te = sc(te, np.full(len(te), bh))
        if want_precal:
            pre_[te] = s_te          # ★ **보정 전** — F1 의 항등이 여기서 정확히 선다
        out[te] = logit(np.clip(cal(s_te), EPS, 1 - EPS))
    return (out, pre_) if want_precal else out

def loro_A(kind, y_override=None):
    out = np.full(len(K), np.nan)
    for held in REC_OK:
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        Ftr = RHY[tr]; fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        ytr = TT_[tr].astype(int) if y_override is None else y_override[held]
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
        sc = lambda ii: lr.decision_function((RHY[ii] - fmu) / fsd)
        cal = make_iso(sc(dv), TT_[dv])
        pi_tr = float(TT_[dv].mean())
        p_te = np.clip(cal(sc(te)), EPS, 1 - EPS)
        l = logit(p_te)
        if kind == "oracle":
            l = l + (logit(BURD[held]) - logit(pi_tr))
        elif kind == "em":
            l = l + (logit(em_prior(p_te, pi_tr)) - logit(pi_tr))
        out[te] = l
    return out

# ── 지표 셋
pooled_of = lambda L: float(average_precision_score(TT_[np.isfinite(L)].astype(int),
                                                    L[np.isfinite(L)]))
def per_rec(L):
    d = {}
    for r in REC_OK:
        pos = IDXS[r]; yy = TT_[pos].astype(int)
        if 0 < yy.sum() < len(yy) and np.all(np.isfinite(L[pos])):
            d[r] = float(average_precision_score(yy, L[pos]))
    return d
macro_of = lambda L: float(np.mean(list(per_rec(L).values())))

def xrec_matrix(L):
    """★★★ **교차레코드 AUROC** — M[i,j] = P(레코드 i 의 양성 > 레코드 j 의 음성).

    매크로가 못 보는 **레코드 간 척도 정렬**만 잰다. 전역 PR-AUC 와 달리 **쌍마다
    동일 가중**이라 양성이 많은 레코드에 안 눌린다(지배 지분 0.152 → 1/56)."""
    P = {r: np.sort(L[IDXS[r]][TT_[IDXS[r]]]) for r in REC_OK}
    N = {r: np.sort(L[IDXS[r]][~TT_[IDXS[r]]]) for r in REC_OK}
    M = np.full((NRE, NRE), np.nan)
    for a, ri in enumerate(REC_OK):
        p = P[ri]
        if not len(p):
            continue
        for b, rj in enumerate(REC_OK):
            if ri == rj:
                continue
            q = N[rj]
            if not len(q):
                continue
            lo = np.searchsorted(q, p, side="left")
            hi = np.searchsorted(q, p, side="right")
            M[a, b] = float((lo + 0.5 * (hi - lo)).sum() / (len(p) * len(q)))
    return M

def xrec_of(M):
    off = ~np.eye(NRE, dtype=bool)
    v = M[off & np.isfinite(M)]
    return float(v.mean()) if len(v) else float("nan")

def xrec_boot(Ms, seed, nb):
    """레코드 군집 부트스트랩 — 같은 재표집을 모든 팔에 태운다.
    같은 원본 레코드끼리의 쌍은 **교차가 아니므로 제외**한다."""
    rng = np.random.RandomState(seed)
    out = {k: [] for k in Ms}
    for _ in range(nb):
        idx = rng.randint(0, NRE, NRE)
        same = idx[:, None] == idx[None, :]
        for k, M in Ms.items():
            sub = M[np.ix_(idx, idx)]
            m = (~same) & np.isfinite(sub)
            out[k].append(float(sub[m].mean()) if m.any() else float("nan"))
    return {k: np.asarray(v, float) for k, v in out.items()}

run.log("  팔 정의 완료 — 모든 팔이 **보정 뒤** 로짓으로 비교된다"
        "(F1 만 **보정 전**을 쓴다)")
CONFIG["cohort"] = dict(n_rec=len(RS), n_ok=NRE, dominant=DOMINANT,
                        dom_rec=DOM_REC, dev_every=DEV_EVERY,
                        xrec_weight=1.0 / NRE)
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【F-A】 팔 실행 · F0 항등 · ★★★ F1 자(보정 전 TT ≡ TS)
run.log("\n" + "=" * 100)
run.log("【F-A】 LORO 실행 · F0 (A 항등) · F1 (보정 전 TT ≡ TS)  ★ 자를 먼저 세운다")
run.log("=" * 100)
T0 = time.time()
perm0 = derangement(NRE, np.random.RandomState(SEED0 + 300))
SHUF = {REC_OK[i]: BURD[REC_OK[perm0[i]]] for i in range(NRE)}
run.log(f"  셔플(derangement) — 자기 자리에 남은 레코드 "
        f"{int(sum(1 for i in range(NRE) if perm0[i] == i))}개(0 이어야 한다)")

L, PRECAL = {}, {}
L["raw"] = loro_A("raw"); run.log(f"  ({time.time()-T0:>5.0f}초) raw 완료")
for k_ in ("oracle", "em"):
    L[f"A_{k_}"] = loro_A(k_); run.log(f"  ({time.time()-T0:>5.0f}초) A_{k_} 완료")
for nm, (tr_s, te_s) in BSPEC.items():
    want = nm in ("TT", "TS")           # F1 에 쓸 두 팔만 보정 전을 남긴다
    r_ = loro_B(tr_s, te_s, SHUF, want_precal=want)
    if want:
        L[nm], PRECAL[nm] = r_
    else:
        L[nm] = r_
    run.log(f"  ({time.time()-T0:>5.0f}초) {nm} (학습 {tr_s} / 테스트 {te_s}) 완료")

POOL = {a: pooled_of(L[a]) for a in ARMS}
PER = {a: per_rec(L[a]) for a in ARMS}
MACRO = {a: float(np.mean(list(PER[a].values()))) for a in ARMS}
T1 = time.time()
XM = {a: xrec_matrix(L[a]) for a in ARMS}
XREC = {a: xrec_of(XM[a]) for a in ARMS}
run.log(f"  ({time.time()-T1:.0f}초) 교차레코드 행렬 {NRE}×{NRE} × {len(ARMS)}팔")

run.log(f"\n  {'팔':<12}{'매크로':>10}{'교차레코드':>12}{'전역(참고)':>12}{'n':>5}")
for a in ARMS:
    run.log(f"  {a:<12}{MACRO[a]:>10.4f}{XREC[a]:>12.4f}{POOL[a]:>12.4f}{len(PER[a]):>5}")
run.log(f"  (Q4-B 앵커 — 매크로 raw {REF['q4b_macro_raw']} · TT {REF['q4b_macro_TT']} · "
        f"TE {REF['q4b_macro_TE']} · SS {REF['q4b_macro_SS']} | 전역 raw "
        f"{REF['q4b_pool_raw']} · A_em {REF['q4b_pool_Aem']} · TT {REF['q4b_pool_TT']})")

# ── F0 — A 팔은 레코드별 상수 시프트라 매크로가 raw 와 **정확히** 같아야 한다
d0 = max(abs(MACRO["A_oracle"] - MACRO["raw"]), abs(MACRO["A_em"] - MACRO["raw"]))
run.log(f"\n  F0 — A 팔 매크로 vs raw 의 max|Δ| = **{d0:.2e}** (허용 {TOL_IDENT:.0e})")
if d0 >= TOL_IDENT:
    raise AssetError(f"F0 실패({d0:.3e}) — A 는 레코드별 상수 시프트라 매크로가 불변이어야 "
                     "한다. 깨졌다면 구현이 틀린 것이므로 아래를 읽지 않는다(R29 ②)")
g_("F0", "✅ 지지", f"A 팔 매크로가 raw 와 **정확히 같다**({d0:.1e}) — "
                    "**A 는 매크로를 못 움직인다**가 구성으로 확인됐다")

# ── ★★★ F1 — **이 런의 자**. 보정 전 `TT` 와 `TS` 의 레코드 내 판별이 정확히 같은가
run.log("\n  ★★★ F1 — **보정 전** `TT` 와 `TS` (테스트시 burden 만 다르다)")
run.log("     burden 은 레코드 내 **상수**이고 선형으로 들어가므로, 테스트시 burden 은")
run.log("     그 레코드의 로짓을 **통째로 평행이동**시킬 뿐 레코드 내 순위를 못 바꾼다.")
pre_TT = {r: float(average_precision_score(TT_[IDXS[r]].astype(int), PRECAL["TT"][IDXS[r]]))
          for r in REC_OK}
pre_TS = {r: float(average_precision_score(TT_[IDXS[r]].astype(int), PRECAL["TS"][IDXS[r]]))
          for r in REC_OK}
d1 = max(abs(pre_TT[r] - pre_TS[r]) for r in REC_OK)
n_same = sum(1 for r in REC_OK
             if np.array_equal(_rank_avg(PRECAL["TT"][IDXS[r]]),
                               _rank_avg(PRECAL["TS"][IDXS[r]])))
m_pre_TT = float(np.mean(list(pre_TT.values())))
m_pre_TS = float(np.mean(list(pre_TS.values())))
run.log(f"     보정 전 매크로 — TT {m_pre_TT:.6f} · TS {m_pre_TS:.6f} · "
        f"max|Δ| **{d1:.2e}**")
run.log(f"     순위 벡터가 완전히 같은 레코드 **{n_same}/{NRE}**")
if d1 >= TOL_IDENT or n_same != NRE:
    raise AssetError(f"F1 실패(max|Δ| {d1:.3e} · 순위 일치 {n_same}/{NRE}) — 보정 전 항등이 "
                     "안 서면 「테스트 몫 ≈ 0」을 등장성 탓으로 돌릴 수 없다. F4 를 읽지 않는다")
g_("F1", "✅ 지지",
   f"보정 전 `TT` ≡ `TS` 가 **정확히** 성립한다(max|Δ| {d1:.1e} · 순위 일치 {n_same}/{NRE}) "
   "— **테스트시 burden 은 레코드 내 판별에 원리적으로 기여하지 않는다**. "
   f"보정 후 남는 차는 전부 **등장성 동점 구조**의 몫이다")
CONFIG["F0"] = dict(macro=MACRO, xrec=XREC, pooled=POOL, ident=float(d0),
                    n_per={a: len(PER[a]) for a in ARMS})
CONFIG["F1"] = dict(pre_macro_TT=m_pre_TT, pre_macro_TS=m_pre_TS,
                    max_abs_diff=float(d1), n_rank_identical=int(n_same), n_rec=NRE)
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【F-B】 F2 매크로 · F3 교차레코드 (공동 주 관문) · 영점
run.log("\n" + "=" * 100)
run.log("【F-B】 F2(매크로) · F3(교차레코드 AUROC) — **공동 주 관문**")
run.log("=" * 100)

T2 = time.time()
BX = xrec_boot({a: XM[a] for a in ARMS}, SEED0 + 11, NB_BOOT)
run.log(f"  ({time.time()-T2:.0f}초) 교차레코드 부트스트랩 {NB_BOOT}회")

def d_macro(a, b, seed):
    ks = [r for r in REC_OK if r in PER[a] and r in PER[b]]
    m_, lo_, hi_, n_ = boot_pair([PER[a][r] for r in ks], [PER[b][r] for r in ks],
                                 seed, NB_BOOT)
    return dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))

def d_xrec(a, b):
    d = BX[b] - BX[a]
    lo_, hi_ = float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
    return dict(mean=XREC[b] - XREC[a], lo=lo_, hi=hi_, mde=float(mde(lo_, hi_)))

F2 = d_macro("A_oracle", "TT", SEED0 + 41)
F3 = d_xrec("A_oracle", "TT")
F5m = d_macro("A_em", "TE", SEED0 + 42)
F5x = d_xrec("A_em", "TE")
run.log(f"\n  {'대비':<26}{'매크로 Δ':>26}{'교차레코드 Δ':>26}")
for nm, dm, dx in (("F2/F3  TT − A_oracle", F2, F3), ("F5     TE − A_em", F5m, F5x)):
    run.log(f"  {nm:<26}{dm['mean']:>+9.4f} [{dm['lo']:+.4f},{dm['hi']:+.4f}]"
            f"{dx['mean']:>+9.4f} [{dx['lo']:+.4f},{dx['hi']:+.4f}]")
run.log(f"    ▸ 성분(R36 ⑤) — 매크로 raw {MACRO['raw']:.4f} · A_or {MACRO['A_oracle']:.4f} · "
        f"TT {MACRO['TT']:.4f} | 교차 raw {XREC['raw']:.4f} · A_or {XREC['A_oracle']:.4f} · "
        f"TT {XREC['TT']:.4f}")

# ── ★★ 대비의 영점 (학습 라벨 치환 · rep × 레코드로 보관한다)
run.log(f"\n  ★★ **대비의 영점** — 학습 라벨 치환 뒤 같은 대비 (reps={N_PERM})")
NUL_MAC, NUL_XM = {}, []
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    la = loro_A("oracle", y_override=yov)
    lb = loro_B("true", "true", SHUF, y_override=yov)
    pa, pb = per_rec(la), per_rec(lb)
    for r in REC_OK:
        if r in pa and r in pb:
            NUL_MAC.setdefault(r, []).append(pb[r] - pa[r])
    Ma, Mb = xrec_matrix(la), xrec_matrix(lb)
    NUL_XM.append((Ma, Mb))
    run.log(f"    ({time.time()-T2:>5.0f}초) 영점 rep {s_+1}/{N_PERM} — "
            f"매크로 {np.mean([pb[r]-pa[r] for r in pa if r in pb]):+.4f} · "
            f"교차 {xrec_of(Mb)-xrec_of(Ma):+.4f}")

NM = boot_mean([float(np.mean(v)) for v in NUL_MAC.values()], SEED0 + 61, NB_BOOT)
_rngN = np.random.RandomState(SEED0 + 62); _xb = []
for _ in range(NB_BOOT):
    idx = _rngN.randint(0, NRE, NRE)
    same = idx[:, None] == idx[None, :]
    vals = []
    for Ma, Mb in NUL_XM:
        sa, sb = Ma[np.ix_(idx, idx)], Mb[np.ix_(idx, idx)]
        m = (~same) & np.isfinite(sa) & np.isfinite(sb)
        if m.any():
            vals.append(float(sb[m].mean() - sa[m].mean()))
    if vals:
        _xb.append(float(np.mean(vals)))
_xb = np.asarray(_xb, float)
NX = ((float(np.mean([xrec_of(Mb) - xrec_of(Ma) for Ma, Mb in NUL_XM])),
       float(np.percentile(_xb, 2.5)), float(np.percentile(_xb, 97.5)), len(_xb))
      if len(_xb) >= 3 else (float("nan"),) * 3 + (len(_xb),))
run.log(f"    매크로   영점 **{NM[0]:+.4f}** [{NM[1]:+.4f}, {NM[2]:+.4f}] (레코드 {NM[3]})")
run.log(f"    교차레코드 영점 **{NX[0]:+.4f}** [{NX[1]:+.4f}, {NX[2]:+.4f}] (부트 {NX[3]})")

# ★ 문턱 규칙 **통일** — 양 관문 모두 max(0, 영점 상단) (Q4-B 는 갈렸다)
NUL_OK = bool(np.isfinite(NM[2]) and np.isfinite(NX[2]))
F2_THR = max(0.0, NM[2]) if np.isfinite(NM[2]) else float("nan")
F3_THR = max(0.0, NX[2]) if np.isfinite(NX[2]) else float("nan")
run.log(f"    ▸ 문턱 = **max(0, 영점 상단)** — 매크로 {F2_THR:+.4f} · 교차 {F3_THR:+.4f} "
        "(양 관문 같은 규칙 · Q4-B 는 갈렸다)")
if not NUL_OK:
    run.log("    ⛔ **영점을 측정하지 못했다** — 0 으로 되돌리는 건 「측정」이 아니라 "
            "「가정」이다(R26). F2·F3·F5 를 **읽지 않는다**")

_w = "" if NUL_OK else " — ★ **영점 미측정**이라 판정 불가(R26)"
def gate(key, d, thr, label, win, lose):
    v = decide(d["lo"], d["hi"], thr, ">") if NUL_OK else "⚠️ 미결"
    if v.startswith("✅"):
        msg = f"{label} — {win} ({d['mean']:+.4f})"
    elif v.startswith("❌"):
        msg = f"{label} — {lose} ({d['mean']:+.4f} · 문턱 {thr:+.4f})"
    else:
        msg = (f"{label} Δ {d['mean']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}] · "
               f"MDE {d['mde']:.4f} · 문턱 {thr:+.4f} — **등가가 아니다**(R33 ①)")
    g_(key, v, msg + _w)
    return v

gate("F2", F2, F2_THR, "매크로",
     "B 가 **레코드 내 판별**을 개선한다(A 가 원리적으로 못 하는 일)",
     "B 가 A 에 못 미친다")
gate("F3", F3, F3_THR, "교차레코드",
     "B 가 **레코드 간 척도 정렬**도 개선한다", "B 가 A 에 못 미친다")
f5m_v = decide(F5m["lo"], F5m["hi"], F2_THR, ">") if NUL_OK else "⚠️ 미결"
f5x_v = decide(F5x["lo"], F5x["hi"], F3_THR, ">") if NUL_OK else "⚠️ 미결"
f5_v = "✅ 지지" if (f5m_v.startswith("✅") and f5x_v.startswith("✅")) else \
       ("❌ 기각" if (f5m_v.startswith("❌") or f5x_v.startswith("❌")) else "⚠️ 미결")
g_("F5", f5_v, f"**배포 가능판끼리** `TE − A_em` — 매크로 {F5m['mean']:+.4f} {f5m_v} · "
                f"교차 {F5x['mean']:+.4f} {f5x_v}. 오라클은 **상한이지 방법이 아니다**"
                f"(전역 참고 — A_em {POOL['A_em']:.4f} vs raw {POOL['raw']:.4f})" + _w)
CONFIG["F2"] = F2; CONFIG["F3"] = F3
CONFIG["F5"] = dict(macro=F5m, xrec=F5x, v_macro=f5m_v, v_xrec=f5x_v)
CONFIG["null_diff"] = dict(macro=dict(mean=NM[0], lo=NM[1], hi=NM[2], n=int(NM[3])),
                           xrec=dict(mean=NX[0], lo=NX[1], hi=NX[2], n=int(NX[3])),
                           n_perm=N_PERM, measured=bool(NUL_OK),
                           f2_thr=float(F2_THR), f3_thr=float(F3_THR))
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【F-C】 ★★★ F4 — 학습 몫 대 테스트 몫 (2×2 분해)
run.log("\n" + "=" * 100)
run.log("【F-C】 ★★★ F4 — **학습 때의 burden 인가, 테스트 때의 burden 인가**")
run.log("=" * 100)
run.log("  Q4-B 의 `B_add_shuf` 는 학습과 테스트를 **동시에** 뒤섞어 두 몫이 안 갈렸다.")
run.log("  빠진 칸 `TS`(학습 true/테스트 shuf)·`ST`(학습 shuf/테스트 true)를 채웠다.")

run.log(f"\n  2×2 표 — 매크로 / 교차레코드 / 전역")
run.log(f"  {'':<16}{'테스트 true':>26}{'테스트 shuf':>26}")
for tr_s, row in (("학습 true", ("TT", "TS")), ("학습 shuf", ("ST", "SS"))):
    cells_ = "".join(f"{MACRO[c]:>8.4f}/{XREC[c]:>7.4f}/{POOL[c]:>7.4f}" for c in row)
    run.log(f"  {tr_s:<16}{cells_}")
run.log(f"  {'(테스트 em)':<16}{MACRO['TE']:>8.4f}/{XREC['TE']:>7.4f}/{POOL['TE']:>7.4f}"
        "   ← 배포 가능판")
run.log(f"  {'raw':<16}{MACRO['raw']:>8.4f}/{XREC['raw']:>7.4f}/{POOL['raw']:>7.4f}")

TRAIN_M = d_macro("SS", "TS", SEED0 + 51)      # 학습 몫 (테스트는 둘 다 shuf)
TEST_M  = d_macro("TS", "TT", SEED0 + 52)      # 테스트 몫 (학습은 둘 다 true)
BOTH_M  = d_macro("SS", "TT", SEED0 + 53)
TRAIN_X = d_xrec("SS", "TS"); TEST_X = d_xrec("TS", "TT"); BOTH_X = d_xrec("SS", "TT")
# 교차 경로 — 가법성이 성립하면 두 경로가 같은 곳에 도착한다
ALT_TR_M = d_macro("ST", "TT", SEED0 + 54)     # 학습 몫 (테스트는 둘 다 true)
ALT_TE_M = d_macro("SS", "ST", SEED0 + 55)     # 테스트 몫 (학습은 둘 다 shuf)

run.log(f"\n  {'분해':<28}{'매크로':>26}{'교차레코드':>26}")
for nm, dm, dx in (("학습 몫  TS − SS", TRAIN_M, TRAIN_X),
                   ("테스트 몫 TT − TS", TEST_M, TEST_X),
                   ("합계     TT − SS", BOTH_M, BOTH_X)):
    run.log(f"  {nm:<28}{dm['mean']:>+9.4f} [{dm['lo']:+.4f},{dm['hi']:+.4f}]"
            f"{dx['mean']:>+9.4f} [{dx['lo']:+.4f},{dx['hi']:+.4f}]")
add_m = TRAIN_M["mean"] + TEST_M["mean"] - BOTH_M["mean"]
add_x = TRAIN_X["mean"] + TEST_X["mean"] - BOTH_X["mean"]
run.log(f"  가법성 잔차 — 매크로 {add_m:+.6f} · 교차 {add_x:+.6f} (0 에 가까워야 한다)")
run.log(f"  대체 경로(학습 몫 TT−ST {ALT_TR_M['mean']:+.4f} · 테스트 몫 ST−SS "
        f"{ALT_TE_M['mean']:+.4f}) — 위와 같은 곳에 도착하는가")

# ★★★ 판정 — 그런데 **분해하기 전에 분해할 것이 있는지부터** 본다.
#    스모크가 잡았다: 합계가 +0.0017(사실상 0)인데도 「학습 몫이다 ✅」가 찍혔고
#    배분이 259% / −159% 로 나왔다. **0 을 쪼개면 아무 비율이나 나온다**(R41 ②).
TOTAL_READABLE = (decide(BOTH_M["lo"], BOTH_M["hi"], 0.0, ">").startswith("✅")
                  and abs(BOTH_M["mean"]) > BOTH_M["mde"])
if TOTAL_READABLE:
    tr_share = TRAIN_M["mean"] / BOTH_M["mean"]
    iso_share = TEST_M["mean"] / BOTH_M["mean"]
    run.log(f"\n  ★ 매크로 이득의 배분 — 학습 몫 **{tr_share:.1%}** · 테스트 몫 {iso_share:.1%}")
else:
    tr_share = iso_share = float("nan")
    run.log(f"\n  ⛔ **배분을 계산하지 않는다** — 쪼갤 합계 자체가 0 과 안 갈린다"
            f"(`TT − SS` {BOTH_M['mean']:+.4f} [{BOTH_M['lo']:+.4f}, {BOTH_M['hi']:+.4f}] · "
            f"MDE {BOTH_M['mde']:.4f}). 0 을 쪼개면 **아무 비율이나 나온다**(R41 ②)")
run.log(f"    (F1 로 **보정 전엔 테스트 몫이 정확히 0** 이므로, 보정 후 테스트 몫은 전부 "
        f"**등장성 동점 구조**의 몫이다. Q4-B 앵커: TE−TT 매크로 −0.0035)")

test_is_zero = (decide(TEST_M["lo"], TEST_M["hi"], 0.0, ">") == "⚠️ 미결"
                and abs(TEST_M["mean"]) < TEST_M["mde"])
train_pos = decide(TRAIN_M["lo"], TRAIN_M["hi"], 0.0, ">").startswith("✅")
if not TOTAL_READABLE:
    f4 = ("⚠️ 미결", f"**분해 불가 — 쪼갤 이득이 없다.** `TT − SS` {BOTH_M['mean']:+.4f} "
                     f"[{BOTH_M['lo']:+.4f}, {BOTH_M['hi']:+.4f}] 가 0 과 안 갈린다. "
                     f"성분(학습 {TRAIN_M['mean']:+.4f} · 테스트 {TEST_M['mean']:+.4f})은 "
                     "**보고만 하고 배분으로 읽지 않는다**(R41 ②)")
elif train_pos and test_is_zero:
    f4 = ("✅ 지지", f"**매크로 이득은 학습 몫이다** — 학습 {TRAIN_M['mean']:+.4f} "
                     f"[{TRAIN_M['lo']:+.4f},{TRAIN_M['hi']:+.4f}] ✅ · 테스트 "
                     f"{TEST_M['mean']:+.4f} [{TEST_M['lo']:+.4f},{TEST_M['hi']:+.4f}] ≈ 0 "
                     f"(합계 {BOTH_M['mean']:+.4f} 의 {tr_share:.0%}) → ★★★ **π̂ 없이 쓸 수 "
                     "있는 처방이다** — Q3 병목을 우회한다")
elif decide(TEST_M["lo"], TEST_M["hi"], 0.0, ">").startswith("✅"):
    f4 = ("❌ 기각", f"**테스트 몫이 0 이 아니다**({TEST_M['mean']:+.4f} "
                     f"[{TEST_M['lo']:+.4f},{TEST_M['hi']:+.4f}] · 합계의 {iso_share:.0%}) — "
                     "F1 로 보정 전엔 정확히 0 이므로 이건 **등장성 보정이 π̂ 오차를 "
                     "증폭**한 것이다. **π̂ 품질이 여전히 병목**이고 Q3 갈래를 다시 연다")
else:
    f4 = ("⚠️ 미결", f"학습 몫 {TRAIN_M['mean']:+.4f} [{TRAIN_M['lo']:+.4f},"
                     f"{TRAIN_M['hi']:+.4f}] · 테스트 몫 {TEST_M['mean']:+.4f} "
                     f"[{TEST_M['lo']:+.4f},{TEST_M['hi']:+.4f}] — 어느 쪽도 0 과 못 갈랐다. "
                     "**등장성 몫의 상한이 남는다**(미결 ≠ 0 · R33 ①)")
g_("F4", f4[0], f4[1])
CONFIG["F4"] = dict(train_macro=TRAIN_M, test_macro=TEST_M, both_macro=BOTH_M,
                    train_xrec=TRAIN_X, test_xrec=TEST_X, both_xrec=BOTH_X,
                    alt_train_macro=ALT_TR_M, alt_test_macro=ALT_TE_M,
                    additivity_macro=float(add_m), additivity_xrec=float(add_x),
                    train_share=float(tr_share), test_share=float(iso_share),
                    total_readable=bool(TOTAL_READABLE),
                    grid={k: dict(macro=MACRO[k], xrec=XREC[k], pooled=POOL[k])
                          for k in BSPEC})
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【F-D】 F6 전역(접는다) · 필요표본 · 진단 · ★ F7 검산표
run.log("\n" + "=" * 100)
run.log("【F-D】 F6 전역(관문 아님) · 필요표본 · 진단 · F7 검산표")
run.log("=" * 100)

# ── F6 — 전역은 **접는다**(닫는 게 아니다 · R36 ①)
dp = POOL["TT"] - POOL["A_oracle"]
run.log(f"  F6 **전역 PR-AUC — 관문 아님**. `TT − A_oracle` = {dp:+.4f} "
        f"(성분 A_or {POOL['A_oracle']:.4f} · TT {POOL['TT']:.4f})")
run.log(f"    Q4-B 실측 {REF['q4b_E3']:+.4f} [{REF['q4b_E3_lo']:+.4f}, "
        f"{REF['q4b_E3_hi']:+.4f}] · MDE {REF['q4b_E3_mde']:.4f}")
run.log(f"    ★ **접는다** — 상한 **{REF['q4b_E3_hi']:+.4f} 이하**. 관문 문턱(0) 기준 "
        f"필요표본이 **{REF['q4b_need_E3_gate']} 레코드**(80%)인데 SVDB 채점 가능 상한은 "
        f"**{NRE}** 다. 유병률이 82배 벌어져 레코드 하나가 양성의 {DOMINANT:.1%} 를 갖는 탓에 "
        "√n 이 안 먹는다 — **코호트가 커지면 재개한다**(R36 ①)")
run.log(f"    ⚠️ Q4-B 는 이 수를 **영점 평균** 기준으로 재서 「32」로 적었다 — 관문이 묻는 "
        "질문과 필요표본이 답하는 질문이 달랐다(R40 ②). 이 런은 **관문 문턱 기준**이다")

run.log(f"\n  필요표본 (**관문 문턱 기준** · 단위 = 레코드 · 현재 {NRE}개)")
run.log(f"  {'대비':<14}{'효과-문턱':>11}{'반폭':>9}{'n(50%)':>9}{'n(80%)':>9}")
NEED = {}
for nm, d, thr in (("F2 매크로", F2, F2_THR), ("F3 교차", F3, F3_THR),
                   ("F5 매크로", F5m, F2_THR), ("F5 교차", F5x, F3_THR)):
    eff = d["mean"] - thr
    n5 = need_super(NRE, d["mde"], eff, False); n8 = need_super(NRE, d["mde"], eff, True)
    if not np.isfinite(eff):
        tag, zero = "★ **영점 미측정** — 계산 불가(R26)", True
    elif abs(eff) < d["mde"]:
        tag, zero = "★ **효과 ≈ 0 이라 해석 불가**(R41 ②)", True
    else:
        tag, zero = ("이미 충분하다" if n8 <= NRE else "읽을 수 있다"), False
    NEED[nm] = dict(effect=float(eff), half=float(d["mde"]), sup50=float(n5),
                    sup80=float(n8), uninterpretable=bool(zero))
    run.log(f"  {nm:<14}{eff:>+11.4f}{d['mde']:>9.4f}{n5:>9.0f}{n8:>9.0f}  {tag}")

run.log(f"\n  ★ 진단 — 지배 레코드 {DOM_REC}(유병률 {BURD[DOM_REC]:.4f} · Q3 의 병목)")
for a in ARMS:
    v = PER[a].get(DOM_REC, float("nan"))
    tag = "  ← A 는 상수 시프트라 raw 와 같아야 한다" if a.startswith("A_") else ""
    run.log(f"  {a:<12}{v:>10.4f}{tag}")
run.log(f"    (Q4-B 앵커 — raw {REF['q4b_dom48_raw']} · TT {REF['q4b_dom48_TT']} · "
        f"TE {REF['q4b_dom48_TE']})")

run.log("\n  ★ F7 — **결론 검산표**")
CHECK = [
    dict(claim=f"F0 항등 — A 팔 매크로가 raw 와 정확히 같다({CONFIG['F0']['ident']:.1e})",
         num="A 는 레코드별 상수 시프트다 — 매크로는 정의상 불변",
         assume="**없음** — 구성으로 보장되고 런타임에 검사한다",
         iffalse="— ★ 이것이 **A 가 매크로를 못 움직인다**는 증명이고, F2 의 의미다"),
    dict(claim=f"F1 자 — 보정 전 `TT` ≡ `TS` (max|Δ| {CONFIG['F1']['max_abs_diff']:.1e} · "
               f"순위 일치 {CONFIG['F1']['n_rank_identical']}/{NRE}) → {VERD['F1']}",
         num="burden 은 레코드 내 상수이고 선형이므로 테스트시 burden 은 순전한 상수 시프트다",
         assume="**없음** — 구성이고 런타임에 검사한다. 깨지면 F4 를 안 읽는다",
         iffalse="★★★ **이게 F4 의 전제다** — 안 서면 「테스트 몫 ≈ 0」을 등장성 탓으로 "
                 "돌릴 수 없다"),
    dict(claim=f"F2 매크로 {F2['mean']:+.4f} [{F2['lo']:+.4f}, {F2['hi']:+.4f}] → {VERD['F2']}",
         num=f"영점 {NM[0]:+.4f} [{NM[1]:+.4f}, {NM[2]:+.4f}] · 문턱 max(0,·) {F2_THR:+.4f} · "
             f"MDE {F2['mde']:.4f} · Q4-B 앵커 {REF['q4b_E2']:+.4f}",
         assume="오라클 burden 이 **레코드 안에서 상수**라는 것",
         iffalse="유병률이 기록 안에서 표류하면 오라클조차 상한이 아니다"),
    dict(claim=f"F3 교차레코드 {F3['mean']:+.4f} [{F3['lo']:+.4f}, {F3['hi']:+.4f}] "
               f"→ {VERD['F3']}",
         num=f"영점 {NX[0]:+.4f} [{NX[1]:+.4f}, {NX[2]:+.4f}] · 문턱 {F3_THR:+.4f} · "
             f"MDE {F3['mde']:.4f} · 쌍마다 동일 가중(지배 {DOMINANT:.3f} → {1.0/NRE:.3f})",
         assume="레코드 쌍이 **교환 가능**하다는 것(군집 부트스트랩의 전제)",
         iffalse="쌍 가중을 동일하게 둔 게 이 지표의 정의다 — 전역과 **다른 것을 잰다**"),
    dict(claim=f"F4 분해 — 학습 몫 {TRAIN_M['mean']:+.4f} · 테스트 몫 {TEST_M['mean']:+.4f} "
               f"(매크로) → {VERD['F4']}",
         num=f"가법성 잔차 {add_m:+.6f} · 대체 경로 학습 {ALT_TR_M['mean']:+.4f} · "
             f"2×2 가 다 찼다(TT/TS/ST/SS)",
         assume="**없음** — F1 이 보정 전 항등을 세웠다",
         iffalse="★ Q4-B 는 `B_add_shuf` 하나로 두 몫을 동시에 뒤섞어 **분해 자체가 "
                 "불가능**했다"),
    dict(claim=f"F5 배포 가능판 — 매크로 {F5m['mean']:+.4f} · 교차 {F5x['mean']:+.4f} "
               f"→ {VERD['F5']}",
         num=f"`A_em` 전역 {POOL['A_em']:.4f} vs raw {POOL['raw']:.4f} "
             f"(Q4-B {REF['q4b_pool_Aem']} vs {REF['q4b_pool_raw']})",
         assume="**없음** — 오라클은 상한이지 방법이 아니다",
         iffalse=f"★ Q4-B 는 이 대비를 「참고」로 뒀다 — 양 지표 CI 가 0 을 뗐는데도"),
    dict(claim="F6 전역은 **접었다**(닫은 게 아니다)",
         num=f"상한 {REF['q4b_E3_hi']:+.4f} · 관문 기준 필요표본 "
             f"{REF['q4b_need_E3_gate']} 레코드 · 가용 {NRE}",
         assume="**없음** — 조건부 재개 조건을 수치로 남긴다",
         iffalse="★ 「미결」을 「등가」로 읽으면 안 된다(R29 ① · R33 ①)"),
    dict(claim=f"영점을 **측정**했다 — {'예' if NUL_OK else '**아니오**'}",
         num=f"rep {N_PERM} × 레코드 {NRE} · 문턱 규칙 **max(0, 영점 상단)** 으로 통일",
         assume="**없음** — 못 쟀으면 문턱을 0 으로 되돌리지 않고 관문을 **읽지 않는다**",
         iffalse="★ Q4-B 는 E2 에 `영점 상단`, E3 에 `max(0, 영점 상단)` 으로 규칙이 "
                 "갈렸다(E2 는 어느 쪽이든 통과해 결론은 안 바뀌었다)"),
]
for i, c in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c['claim']}**")
    run.log(f"      근거   {c['num']}")
    run.log(f"      가정   {c['assume']}")
    run.log(f"      틀리면 {c['iffalse']}")
CONFIG["F6"] = dict(pooled_diff=float(dp), pooled=POOL, folded=True,
                    upper=REF["q4b_E3_hi"], need_gate=REF["q4b_need_E3_gate"])
CONFIG["need"] = NEED; CONFIG["F7"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【F-E】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
EN = {"raw": "raw", "A_oracle": "A or", "A_em": "A em", "TT": "TT", "TE": "TE (deploy)",
      "TS": "TS", "ST": "ST", "SS": "SS"}
xs = np.arange(len(ARMS))
ax[0].bar(xs - 0.22, [MACRO[a] for a in ARMS], width=0.44, color="tab:green", label="macro")
ax[0].bar(xs + 0.22, [XREC[a] for a in ARMS], width=0.44, color="tab:orange", label="cross-record")
ax[0].axhline(MACRO["raw"], ls="--", color="k", lw=1.0)
ax[0].set_xticks(xs); ax[0].set_xticklabels([EN[a] for a in ARMS], fontsize=7, rotation=20)
ax[0].set_ylabel("score")
ax[0].set_title(f"arms (LORO, n={NRE}) — pooled is NOT gated", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3, axis="y")

nm = ["F2 macro", "F3 x-rec", "F4 train (m)", "F4 test (m)", "F5 macro", "F5 x-rec"]
DD = [F2, F3, TRAIN_M, TEST_M, F5m, F5x]
vv = [d["mean"] for d in DD]
lo = [d["mean"] - d["lo"] for d in DD]; hi = [d["hi"] - d["mean"] for d in DD]
cols = ["tab:red", "tab:red", "tab:blue", "tab:blue", "tab:purple", "tab:purple"]
for i, (v, l_, h_, c_) in enumerate(zip(vv, lo, hi, cols)):
    ax[1].errorbar([v], [i], xerr=[[l_], [h_]], fmt="o", capsize=5, color=c_)
ax[1].axvline(0, color="k", lw=.9)
if np.isfinite(F2_THR):
    ax[1].axvline(F2_THR, ls=":", color="tab:gray", lw=1.2, label=f"macro thr {F2_THR:+.3f}")
else:
    ax[1].plot([], [], " ", label="thr: null NOT measured")
ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels(nm, fontsize=8)
ax[1].set_xlabel("paired difference")
ax[1].set_title("F4 : train share vs test share", fontsize=9)
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis="x")

ks = sorted(PER["A_oracle"].keys())
ax[2].scatter([PER["A_oracle"][r] for r in ks], [PER["TT"][r] for r in ks],
              s=26, color="tab:purple", label="TT")
ax[2].scatter([PER["A_oracle"][r] for r in ks], [PER["TE"][r] for r in ks],
              s=18, color="tab:cyan", marker="^", label="TE (deploy)")
ax[2].plot([0, 1], [0, 1], "k--", lw=.9)
ax[2].set_xlabel("A oracle (per record)"); ax[2].set_ylabel("B (per record)")
ax[2].set_title("macro is where A cannot move", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q4c_burden_train_vs_test", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  LORO {NRE}레코드 (매크로/교차/전역) — " +
        " · ".join(f"{a} {MACRO[a]:.4f}/{XREC[a]:.4f}/{POOL[a]:.4f}" for a in ARMS))
run.log(f"  영점 — 매크로 {NM[0]:+.4f} [{NM[1]:+.4f}, {NM[2]:+.4f}] · "
        f"교차 {NX[0]:+.4f} [{NX[1]:+.4f}, {NX[2]:+.4f}]")
run.log("")
if not NUL_OK:
    run.log("  ⛔ **판정 보류 — 대비의 영점을 측정하지 못했다.**")
    run.log("     문턱을 0 으로 되돌리는 건 「측정」이 아니라 「가정」이다(R26).")
elif ok_("F2") and ok_("F3"):
    run.log("  ★★★ **B 가 레코드 내·레코드 간 모두 A 를 이긴다.**")
    run.log(f"     매크로 {F2['mean']:+.4f} (문턱 {F2_THR:+.4f}) · "
            f"교차레코드 {F3['mean']:+.4f} (문턱 {F3_THR:+.4f})")
    run.log(f"     배포 가능판 {VERD['F5']} — 매크로 {F5m['mean']:+.4f} · 교차 {F5x['mean']:+.4f}")
elif ok_("F2"):
    run.log("  ★★ **B 의 이득은 레코드 내 판별에 있다 — A 가 원리적으로 못 하는 일이다.**")
    run.log(f"     매크로 {F2['mean']:+.4f} [{F2['lo']:+.4f}, {F2['hi']:+.4f}] "
            f"(문턱 {F2_THR:+.4f})")
    run.log(f"     교차레코드는 {VERD['F3']} — {F3['mean']:+.4f} "
            f"[{F3['lo']:+.4f}, {F3['hi']:+.4f}] · 필요표본 "
            f"{NEED['F3 교차']['sup80']:.0f}(80% · 현재 {NRE})")
elif no_("F2"):
    run.log("  ⛔ **B 가 A 에 못 미친다** — 매크로 CI 상단이 문턱 아래다.")
    run.log(f"     매크로 {F2['mean']:+.4f} [{F2['lo']:+.4f}, {F2['hi']:+.4f}] vs {F2_THR:+.4f}")
else:
    run.log("  ⚠️ **미결 — 가르지 못했다.** 「A 와 등가」가 **아니다**(R29 ① · R33 ①).")
    run.log(f"     매크로 {F2['mean']:+.4f} [{F2['lo']:+.4f}, {F2['hi']:+.4f}] · "
            f"MDE {F2['mde']:.4f} vs 문턱 {F2_THR:+.4f}")
    run.log(f"     필요표본(레코드) — 매크로 {NEED['F2 매크로']['sup80']:.0f} · "
            f"교차 {NEED['F3 교차']['sup80']:.0f} (80% · 현재 {NRE})")
run.log("")
run.log(f"  ★★★ **F4 분해** — 매크로 합계 {BOTH_M['mean']:+.4f} "
        f"[{BOTH_M['lo']:+.4f}, {BOTH_M['hi']:+.4f}] · 학습 몫 **{TRAIN_M['mean']:+.4f}** · "
        f"테스트 몫 **{TEST_M['mean']:+.4f}**"
        + ("" if TOTAL_READABLE else "  ⛔ 합계가 0 과 안 갈려 **배분으로 읽지 않는다**"))
run.log(f"     → {VERD['F4']}  {NOTE['F4']}")
run.log(f"  ★★ **F6 전역은 접었다** — 상한 {REF['q4b_E3_hi']:+.4f} 이하 · 관문 기준 "
        f"필요표본 {REF['q4b_need_E3_gate']} 레코드 vs 가용 {NRE}. **닫은 게 아니다**(R36 ①)")
run.log(f"  ★ 지배 레코드 {DOM_REC} — raw {PER['raw'].get(DOM_REC, float('nan')):.4f} → "
        f"TT {PER['TT'].get(DOM_REC, float('nan')):.4f} · "
        f"TE {PER['TE'].get(DOM_REC, float('nan')):.4f} (Q3 의 병목 레코드)")

run.finish({
    "exp_id": "quest46_q4c_burden_train_vs_test",
    "metric": "train_share_of_macro_gain",
    "value": float(TRAIN_M["mean"]),
    "passed": bool(ok_("F0") and ok_("F1") and ok_("F2")),
    "summary": ("Q4-B 가 남긴 셋을 풀었다 — ① `B_add_shuf` 가 학습·테스트 burden 을 동시에 "
                "뒤섞어 분해가 불가능했던 걸 2×2(TT/TS/ST/SS)로 채워 **학습 몫과 테스트 몫을 "
                "갈랐다**(F1 이 보정 전 항등으로 자를 먼저 세운다) ② 전역 PR-AUC 는 SVDB 에서 "
                "구조적으로 검정력이 없어(관문 기준 356 레코드 · 가용 56) **관문에서 내리고 "
                "교차레코드 AUROC(쌍마다 동일 가중)로 갈아탔다** ③ 배포 가능판끼리"
                "(`TE − A_em`)를 관문으로 승격했다. Q4-B 의 오류 둘(필요표본을 영점 평균 "
                "기준으로 계산 · 문턱 규칙 불일치)도 정정했다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}),
    "F0": CONFIG.get("F0", {}), "F1": CONFIG.get("F1", {}), "F2": CONFIG.get("F2", {}),
    "F3": CONFIG.get("F3", {}), "F4": CONFIG.get("F4", {}), "F5": CONFIG.get("F5", {}),
    "F6": CONFIG.get("F6", {}), "null_diff": CONFIG.get("null_diff", {}),
    "need": CONFIG.get("need", {}), "F7": CONFIG.get("F7", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4c_burden_train_vs_test.ipynb`")
